# 06 — Team Tactical Style Analysis

This notebook extends the project from **player-level valuation** to **team-level tactical intelligence**.

## Objectives

1. Load action-level outputs from the previous notebooks.
2. Build a team tactical dataset combining:
   - action volume
   - progression structure
   - xT contribution
   - VAEP contribution
   - spatial usage patterns
3. Quantify how teams accumulate value through:
   - passing
   - carries
   - dribbles
   - progressive actions
4. Compare teams through:
   - style metrics
   - efficiency metrics
   - tactical clusters
5. Produce visual summaries of team styles.
6. Persist the final team tactical table for downstream reporting.

## Outputs

- `data/processed/team_tactical_table.parquet`
- `data/processed/team_style_clusters.parquet`
- `db/football_value.duckdb`
  - `team_tactical_table`
  - `team_style_clusters`


## 0. Setup

In [ ]:
from __future__ import annotations

import warnings
from dataclasses import dataclass
from pathlib import Path
from typing import Optional

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 180)
warnings.filterwarnings("ignore")


In [ ]:
try:
    import duckdb
except Exception as e:
    raise ImportError("duckdb is required. Install it with: pip install duckdb") from e

try:
    from sklearn.preprocessing import StandardScaler
    from sklearn.cluster import KMeans
    SKLEARN_AVAILABLE = True
except Exception:
    SKLEARN_AVAILABLE = False

try:
    from mplsoccer import Pitch
    MPLSOCCER_AVAILABLE = True
except Exception:
    MPLSOCCER_AVAILABLE = False


## 1. Project paths

In [ ]:
def find_repo_root(start: Optional[Path] = None) -> Path:
    start = start or Path.cwd()
    for p in [start, *start.parents]:
        if (p / "README.md").exists():
            return p
    return start

REPO_ROOT = find_repo_root()
DATA_DIR = REPO_ROOT / "data"
PROCESSED_DIR = DATA_DIR / "processed"
DB_DIR = REPO_ROOT / "db"
REPORTS_DIR = REPO_ROOT / "reports"

for d in [PROCESSED_DIR, DB_DIR, REPORTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

DB_PATH = DB_DIR / "football_value.duckdb"

ACTIONS_PATH = PROCESSED_DIR / "actions.parquet"
ACTIONS_XT_PATH = PROCESSED_DIR / "actions_xt.parquet"
ACTIONS_VAEP_PATH = PROCESSED_DIR / "actions_vaep.parquet"
TEAM_XT_PATH = PROCESSED_DIR / "team_xt_summary.parquet"

TEAM_TACTICAL_PATH = PROCESSED_DIR / "team_tactical_table.parquet"
TEAM_STYLE_CLUSTERS_PATH = PROCESSED_DIR / "team_style_clusters.parquet"

REPO_ROOT, DB_PATH


## 2. Configuration

In [ ]:
@dataclass(frozen=True)
class Config:
    pitch_length: float = 120.0
    pitch_width: float = 80.0
    n_x_bins: int = 12
    n_y_bins: int = 8
    n_clusters: int = 4
    random_state: int = 42
    save_outputs: bool = True

CFG = Config()
CFG


## 3. Load required datasets

Expected inputs:
- `actions.parquet`
- `actions_xt.parquet`
- `actions_vaep.parquet`
- `team_xt_summary.parquet`


In [ ]:
required_paths = [
    ACTIONS_PATH,
    ACTIONS_XT_PATH,
    ACTIONS_VAEP_PATH,
    TEAM_XT_PATH,
]

missing_paths = [str(p) for p in required_paths if not p.exists()]
if missing_paths:
    raise FileNotFoundError(
        "Missing required inputs. Run Notebooks 02, 03, and 04 first:\n" + "\n".join(missing_paths)
    )

actions = pd.read_parquet(ACTIONS_PATH)
actions_xt = pd.read_parquet(ACTIONS_XT_PATH)
actions_vaep = pd.read_parquet(ACTIONS_VAEP_PATH)
team_xt_summary = pd.read_parquet(TEAM_XT_PATH)

print("actions:", actions.shape)
print("actions_xt:", actions_xt.shape)
print("actions_vaep:", actions_vaep.shape)
print("team_xt_summary:", team_xt_summary.shape)


## 4. Build a unified team action table

In [ ]:
team_actions = (
    actions
    .merge(
        actions_xt[["event_id", "xt_added", "xt_start", "xt_end"]],
        on="event_id",
        how="left"
    )
    .merge(
        actions_vaep[["event_id", "p_score_next_k_actions", "p_concede_next_k_actions", "vaep_value"]],
        on="event_id",
        how="left"
    )
)

team_actions["xt_added"] = team_actions["xt_added"].fillna(0.0)
team_actions["vaep_value"] = team_actions["vaep_value"].fillna(0.0)
team_actions["p_score_next_k_actions"] = team_actions["p_score_next_k_actions"].fillna(0.0)
team_actions["p_concede_next_k_actions"] = team_actions["p_concede_next_k_actions"].fillna(0.0)

team_actions.head()


## 5. Build base team volume table

In [ ]:
team_base = (
    team_actions.groupby("team", dropna=False)
    .agg(
        n_actions=("event_id", "count"),
        n_matches=("match_id", "nunique"),
        passes=("is_pass", "sum"),
        carries=("is_carry", "sum"),
        dribbles=("is_dribble", "sum"),
        shots=("is_shot", "sum"),
        progressive_actions=("is_progressive_action", "sum"),
        total_xt=("xt_added", "sum"),
        total_vaep=("vaep_value", "sum"),
        avg_xt=("xt_added", "mean"),
        avg_vaep=("vaep_value", "mean"),
        total_p_score=("p_score_next_k_actions", "sum"),
        total_p_concede=("p_concede_next_k_actions", "sum"),
    )
    .reset_index()
)

team_base["pseudo_minutes"] = team_base["n_matches"] * 96.0

team_base["actions_per_90"] = np.where(
    team_base["pseudo_minutes"] > 0,
    90 * team_base["n_actions"] / team_base["pseudo_minutes"],
    np.nan
)

team_base["progressive_actions_per_90"] = np.where(
    team_base["pseudo_minutes"] > 0,
    90 * team_base["progressive_actions"] / team_base["pseudo_minutes"],
    np.nan
)

team_base["xt_per_90"] = np.where(
    team_base["pseudo_minutes"] > 0,
    90 * team_base["total_xt"] / team_base["pseudo_minutes"],
    np.nan
)

team_base["vaep_per_90"] = np.where(
    team_base["pseudo_minutes"] > 0,
    90 * team_base["total_vaep"] / team_base["pseudo_minutes"],
    np.nan
)

team_base.head()


## 6. Derive tactical mix and efficiency metrics

In [ ]:
team_base["pass_share"] = team_base["passes"] / team_base["n_actions"]
team_base["carry_share"] = team_base["carries"] / team_base["n_actions"]
team_base["dribble_share"] = team_base["dribbles"] / team_base["n_actions"]
team_base["shot_share"] = team_base["shots"] / team_base["n_actions"]

team_base["xt_per_100_actions"] = 100 * team_base["total_xt"] / team_base["n_actions"]
team_base["vaep_per_100_actions"] = 100 * team_base["total_vaep"] / team_base["n_actions"]

team_base["xt_per_progressive_action"] = np.where(
    team_base["progressive_actions"] > 0,
    team_base["total_xt"] / team_base["progressive_actions"],
    np.nan
)

team_base["vaep_per_progressive_action"] = np.where(
    team_base["progressive_actions"] > 0,
    team_base["total_vaep"] / team_base["progressive_actions"],
    np.nan
)

team_base["attack_risk_balance"] = team_base["total_p_score"] - team_base["total_p_concede"]

team_base.head()


## 7. Build spatial tactical features

We quantify how teams use different pitch channels and field zones.

This provides a compact tactical fingerprint based on action origins.


In [ ]:
spatial = team_actions.copy()

# Channel definitions
spatial["is_left_channel"] = (spatial["y_start"] < CFG.pitch_width / 3).astype(int)
spatial["is_central_channel"] = ((spatial["y_start"] >= CFG.pitch_width / 3) & (spatial["y_start"] <= 2 * CFG.pitch_width / 3)).astype(int)
spatial["is_right_channel"] = (spatial["y_start"] > 2 * CFG.pitch_width / 3).astype(int)

# Vertical thirds
spatial["is_defensive_third"] = (spatial["x_start"] < CFG.pitch_length / 3).astype(int)
spatial["is_middle_third"] = ((spatial["x_start"] >= CFG.pitch_length / 3) & (spatial["x_start"] <= 2 * CFG.pitch_length / 3)).astype(int)
spatial["is_attacking_third"] = (spatial["x_start"] > 2 * CFG.pitch_length / 3).astype(int)

team_spatial = (
    spatial.groupby("team", dropna=False)
    .agg(
        left_channel_actions=("is_left_channel", "sum"),
        central_channel_actions=("is_central_channel", "sum"),
        right_channel_actions=("is_right_channel", "sum"),
        defensive_third_actions=("is_defensive_third", "sum"),
        middle_third_actions=("is_middle_third", "sum"),
        attacking_third_actions=("is_attacking_third", "sum"),
        mean_x_start=("x_start", "mean"),
        mean_y_start=("y_start", "mean"),
        mean_delta_x=("delta_x", "mean"),
        mean_move_length=("move_length", "mean"),
    )
    .reset_index()
)

for col in [
    "left_channel_actions", "central_channel_actions", "right_channel_actions",
    "defensive_third_actions", "middle_third_actions", "attacking_third_actions"
]:
    team_spatial[f"{col}_share"] = team_spatial[col] / (
        team_spatial["left_channel_actions"] +
        team_spatial["central_channel_actions"] +
        team_spatial["right_channel_actions"]
    ).replace(0, np.nan)

team_spatial.head()


## 8. Build the integrated team tactical table

In [ ]:
team_tactical = (
    team_base
    .merge(
        team_spatial[[
            "team",
            "left_channel_actions_share",
            "central_channel_actions_share",
            "right_channel_actions_share",
            "defensive_third_actions_share",
            "middle_third_actions_share",
            "attacking_third_actions_share",
            "mean_x_start",
            "mean_y_start",
            "mean_delta_x",
            "mean_move_length",
        ]],
        on="team",
        how="left"
    )
    .merge(
        team_xt_summary.rename(columns={
            "total_xt_added": "team_xt_total_from_xt_table",
            "xt_added_per_100_actions": "team_xt_per_100_from_xt_table",
        }),
        on="team",
        how="left",
        suffixes=("", "_xt")
    )
)

team_tactical.head()


## 9. Percentile-based tactical indicators

In [ ]:
percentile_cols = [
    "actions_per_90",
    "progressive_actions_per_90",
    "xt_per_90",
    "vaep_per_90",
    "xt_per_100_actions",
    "vaep_per_100_actions",
    "mean_delta_x",
    "mean_move_length",
    "central_channel_actions_share",
    "middle_third_actions_share",
    "attacking_third_actions_share",
]

for col in percentile_cols:
    team_tactical[f"{col}_pct"] = team_tactical[col].rank(pct=True)

team_tactical["progression_style_score"] = (
    0.40 * team_tactical["progressive_actions_per_90_pct"]
    + 0.30 * team_tactical["xt_per_90_pct"]
    + 0.30 * team_tactical["mean_delta_x_pct"]
)

team_tactical["control_style_score"] = (
    0.35 * team_tactical["actions_per_90_pct"]
    + 0.35 * team_tactical["central_channel_actions_share_pct"]
    + 0.30 * team_tactical["middle_third_actions_share_pct"]
)

team_tactical["directness_score"] = (
    0.50 * team_tactical["mean_delta_x_pct"]
    + 0.50 * team_tactical["mean_move_length_pct"]
)

team_tactical["final_third_value_score"] = (
    0.50 * team_tactical["attacking_third_actions_share_pct"]
    + 0.50 * team_tactical["vaep_per_90_pct"]
)

team_tactical.head()

## 10. Rule-based tactical style labels

In [ ]:
def assign_team_style(row):
    if row["control_style_score"] >= 0.75 and row["central_channel_actions_share"] >= 0.35:
        return "Control-possession side"
    if row["directness_score"] >= 0.75 and row["mean_delta_x"] > 10:
        return "Direct progression side"
    if row["progression_style_score"] >= 0.75 and row["xt_per_90"] >= team_tactical["xt_per_90"].median():
        return "Progressive value side"
    if row["final_third_value_score"] >= 0.75:
        return "Final-third efficient side"
    return "Balanced tactical profile"

team_tactical["rule_based_style"] = team_tactical.apply(assign_team_style, axis=1)
team_tactical["rule_based_style"].value_counts()


## 11. Optional clustering of tactical styles

In [ ]:
if SKLEARN_AVAILABLE:
    cluster_features = [
        "actions_per_90",
        "progressive_actions_per_90",
        "xt_per_90",
        "vaep_per_90",
        "pass_share",
        "carry_share",
        "dribble_share",
        "left_channel_actions_share",
        "central_channel_actions_share",
        "right_channel_actions_share",
        "mean_delta_x",
        "mean_move_length",
    ]

    cluster_df = team_tactical[cluster_features].fillna(0).copy()
    scaler = StandardScaler()
    X_cluster = scaler.fit_transform(cluster_df)

    kmeans = KMeans(n_clusters=CFG.n_clusters, random_state=CFG.random_state, n_init=10)
    team_tactical["cluster_id"] = kmeans.fit_predict(X_cluster)
else:
    team_tactical["cluster_id"] = -1

team_tactical[["team", "rule_based_style", "cluster_id"]].head(20)


## 12. Build team style clusters output

In [ ]:
team_style_clusters = team_tactical[[
    "team",
    "n_actions",
    "n_matches",
    "actions_per_90",
    "progressive_actions_per_90",
    "xt_per_90",
    "vaep_per_90",
    "pass_share",
    "carry_share",
    "dribble_share",
    "central_channel_actions_share",
    "attacking_third_actions_share",
    "mean_delta_x",
    "mean_move_length",
    "progression_style_score",
    "control_style_score",
    "directness_score",
    "final_third_value_score",
    "rule_based_style",
    "cluster_id",
]].copy()

team_style_clusters = team_style_clusters.sort_values("vaep_per_90", ascending=False).reset_index(drop=True)
team_style_clusters.head(20)


## 13. Tactical comparison plots

In [ ]:
fig, ax = plt.subplots(figsize=(10, 7))
ax.scatter(
    team_tactical["xt_per_90"],
    team_tactical["vaep_per_90"],
    alpha=0.8
)

ax.set_title("Team Tactical Value Map: xT per 90 vs VAEP per 90")
ax.set_xlabel("xT per 90")
ax.set_ylabel("VAEP per 90")

for _, r in team_tactical.iterrows():
    ax.annotate(r["team"], (r["xt_per_90"], r["vaep_per_90"]), fontsize=8)

plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(10, 7))
ax.scatter(
    team_tactical["mean_delta_x"],
    team_tactical["pass_share"],
    alpha=0.8
)

ax.set_title("Team Style Map: Directness vs Pass Share")
ax.set_xlabel("Mean Delta X")
ax.set_ylabel("Pass Share")

for _, r in team_tactical.iterrows():
    ax.annotate(r["team"], (r["mean_delta_x"], r["pass_share"]), fontsize=8)

plt.show()


## 14. Top tactical rankings

In [ ]:
top_progression_teams = team_tactical.sort_values("progression_style_score", ascending=False)[[
    "team", "progressive_actions_per_90", "xt_per_90", "mean_delta_x", "progression_style_score", "rule_based_style"
]].head(20)

top_progression_teams


In [ ]:
top_control_teams = team_tactical.sort_values("control_style_score", ascending=False)[[
    "team", "actions_per_90", "central_channel_actions_share", "middle_third_actions_share", "control_style_score", "rule_based_style"
]].head(20)

top_control_teams


In [ ]:
top_final_third_teams = team_tactical.sort_values("final_third_value_score", ascending=False)[[
    "team", "attacking_third_actions_share", "vaep_per_90", "final_third_value_score", "rule_based_style"
]].head(20)

top_final_third_teams


## 15. Optional radar-style team comparison

In [ ]:
def radar_plot(team_names, metrics, df):
    subset = df[df["team"].isin(team_names)].copy()
    if subset.empty or len(metrics) < 3:
        print("Not enough data for radar plot.")
        return

    angles = np.linspace(0, 2 * np.pi, len(metrics), endpoint=False).tolist()
    angles += angles[:1]

    fig = plt.figure(figsize=(8, 8))
    ax = plt.subplot(111, polar=True)

    for _, row in subset.iterrows():
        values = [row[m] for m in metrics]
        values += values[:1]
        ax.plot(angles, values, linewidth=2, label=row["team"])
        ax.fill(angles, values, alpha=0.10)

    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(metrics)
    ax.set_title("Team Tactical Comparison Radar")
    ax.legend(loc="upper right", bbox_to_anchor=(1.35, 1.10))
    plt.show()

radar_metrics = [
    "actions_per_90_pct",
    "progressive_actions_per_90_pct",
    "xt_per_90_pct",
    "vaep_per_90_pct",
    "directness_score",
]

example_teams = team_tactical.sort_values("vaep_per_90", ascending=False)["team"].head(3).tolist()
radar_plot(example_teams, radar_metrics, team_tactical)


## 16. Save outputs

In [ ]:
if CFG.save_outputs:
    team_tactical.to_parquet(TEAM_TACTICAL_PATH, index=False)
    team_style_clusters.to_parquet(TEAM_STYLE_CLUSTERS_PATH, index=False)

    try:
        con.close()
    except:
        pass

    con = duckdb.connect(str(DB_PATH))
    con.execute("CREATE OR REPLACE TABLE team_tactical_table AS SELECT * FROM read_parquet(?)", [str(TEAM_TACTICAL_PATH)])
    con.execute("CREATE OR REPLACE TABLE team_style_clusters AS SELECT * FROM read_parquet(?)", [str(TEAM_STYLE_CLUSTERS_PATH)])
    con.close()

    print(f"Saved team tactical table to: {TEAM_TACTICAL_PATH}")
    print(f"Saved team style clusters to: {TEAM_STYLE_CLUSTERS_PATH}")
    print(f"Updated DuckDB at:            {DB_PATH}")


## 17. Preview from DuckDB

In [ ]:
con = duckdb.connect(str(DB_PATH))

summary_sql = '''
SELECT
    COUNT(*) AS n_teams,
    AVG(xt_per_90) AS avg_xt_per_90,
    AVG(vaep_per_90) AS avg_vaep_per_90,
    AVG(progression_style_score) AS avg_progression_style_score,
    AVG(control_style_score) AS avg_control_style_score
FROM team_tactical_table
'''
preview = con.execute(summary_sql).df()
con.close()

preview


# Conclusions

## Team Tactical Dataset

The team tactical analysis was built on:

* **158,114 on-ball actions**
* **51 teams**
* integrated outputs from:

  * raw action modelling
  * **xT**
  * **VAEP-style action valuation**

This creates a team-level tactical layer that combines action volume, progression structure, spatial usage, and value creation.

---

## Tactical Style Distribution

The rule-based team segmentation produced the following distribution:

| Style                      | Teams |
| -------------------------- | ----: |
| Balanced tactical profile  |    37 |
| Final-third efficient side |     7 |
| Direct progression side    |     4 |
| Control-possession side    |     3 |

This distribution is realistic for mixed open football datasets, where most teams tend to fall into broad balanced profiles, while only a smaller subset shows highly distinctive tactical identities.

---

## High-Value Teams

The top teams by total tactical value and per-90 attacking output include:

* **Germany**
* **Brazil**
* **Spain**
* **Barcelona**
* **Argentina**

These teams show the strongest combination of:

* high action volume
* strong progressive output
* high xT generation
* high VAEP contribution

This is consistent with historically dominant or high-quality possession sides.

---

## Final-Third Efficiency

The strongest **final-third efficient sides** include:

* **Germany**
* **Spain**
* **Portugal**
* **Brazil**
* **Argentina**

These teams combine:

* relatively high attacking-third action share
* strong VAEP per 90
* effective value creation close to goal

This indicates that they are not only progressing the ball well, but also converting possession into meaningful attacking threat efficiently.

---

## Control and Possession Profiles

The strongest **control-possession profiles** include:

* **Barcelona**
* **Real Madrid**
* **Uruguay**

These teams stand out through:

* high actions per 90
* high central channel usage
* strong middle-third occupation

This profile reflects structured possession play, territorial control, and higher involvement in central circulation phases.

---

## Progression-Oriented Teams

The strongest **progression style teams** include:

* **Cádiz**
* **Croatia**
* **Germany**
* **Barcelona**
* **Brazil**
* **Spain**

These teams rank highly on:

* progressive actions per 90
* xT per 90
* mean forward ball progression (`mean_delta_x`)

This suggests that they advance possession effectively either through controlled progression or more direct field gain.

---

## Spatial Style Interpretation

The **Directness vs Pass Share** map shows a meaningful spread across teams:

* some teams combine **high pass share with low directness**, indicating more patient circulation
* others show **higher mean forward gain**, indicating more vertical progression
* a few teams operate with both meaningful verticality and strong passing structure

This is useful for distinguishing:

* possession-heavy teams
* vertical progression teams
* mixed transitional profiles

---

## Team Value Map

The **xT per 90 vs VAEP per 90** comparison highlights clear tactical separation between teams.

Notable observations:

* **Germany** stands out as the strongest overall attacking-value team in this sample
* **Brazil, Spain, and Barcelona** also occupy the high-value quadrant
* several teams produce moderate xT but lower VAEP, suggesting weaker conversion of progression into short-term outcome value
* others produce strong VAEP relative to xT, indicating efficient attacking exploitation

This comparison is particularly useful for identifying whether a team’s value comes primarily from:

* territorial progression
* efficient end-product
* or a combination of both

---

## Tactical Intelligence Value

This notebook moves the project beyond player valuation into **team tactical intelligence**.

The resulting team tactical table supports:

* team style benchmarking
* opposition profiling
* contextual scouting
* identifying the environments in which players may fit best

This is especially valuable because player recruitment should be interpreted relative to **team playing style**, not only player-level outputs.

---

## Project Significance

At this stage, the project includes a full multi-layer football analytics pipeline:

```text
StatsBomb event data
→ event modelling
→ xT valuation
→ VAEP-style action valuation
→ player value analysis
→ team tactical style analysis
```

This is already a strong professional portfolio structure because it combines:

* reproducible data engineering
* football action valuation
* player profiling
* tactical team intelligence

---

## Next Step

The next logical step is to build a **player-role and tactical fit layer**, linking:

* player progression and value profiles
* team tactical style fingerprints

This will turn the project into a more explicit **recruitment decision-support framework**.
